In [0]:
claims = spark.table("sdp_catalog.silver.claims_clean").drop("city")

customers = spark.table("sdp_catalog.silver.customers_clean")

policies = spark.table("sdp_catalog.silver.policies_clean")

fact_claims = claims \
.join(customers, "customer_id") \
.join(policies, "policy_id")

# 🚀 PASO 17 — KPI resolución

In [0]:
from pyspark.sql.functions import datediff, col

fact_claims = fact_claims.withColumn(
    "resolution_days",
    datediff(
        col("close_date"),
        col("claim_date")
    )
)

# 🚀 PASO 18 — Fraud Flag

In [0]:
from pyspark.sql.functions import when
fact_claims = fact_claims.withColumn(
    "fraud_flag",
    when(col("claim_amount") > 20000, 1)
    .otherwise(0)
)

In [0]:
spark.sql("DROP TABLE IF EXISTS sdp_catalog.gold.fact_claims")

fact_claims.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("sdp_catalog.gold.fact_claims")